# Demo gợi ý technical skill

Notebook này chỉ tải model đã lưu, không chạy lại bước khai phá luật hay 5-fold. Chọn role, cấp kinh nghiệm, những skill đã biết và số lượng gợi ý; model ưu tiên luật Apriori rồi dùng phổ biến theo role–kinh nghiệm để lấp danh sách còn thiếu.

## 1. Tải model

In [1]:
from pathlib import Path
import joblib
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

MODEL_PATH = Path("../models/skill_recommender.joblib")
model = joblib.load(MODEL_PATH)
print("Đã tải:", MODEL_PATH.resolve())
print("Thuật toán:", model["algorithm"], "| số luật:", sum(map(len, model["rule_index"].values())))


Đã tải: D:\DataMining\skill-recommendation-using-association-rules\models\skill_recommender.joblib
Thuật toán: Apriori | số luật: 26074


## 2. Tạo hàm gợi ý

Luật chỉ được dùng khi toàn bộ skill ở antecedent nằm trong danh sách skill đã biết. Skill đã biết không được trả lại. Nếu luật tìm chưa đủ K skill mới, danh sách được bổ sung từ nhóm role và exp đang chọn.

In [2]:
def recommend(role, exp_level, known_skills, k=10):
    role_exp = f"{role} | {exp_level}"
    known = set(known_skills)
    candidates = []
    for antecedent, skill, confidence, specificity, lift, support_count in model["rule_index"].get(role_exp, []):
        if antecedent.issubset(known) and skill not in known:
            candidates.append((skill, confidence, specificity, lift, support_count))
    best = {}
    for skill, confidence, specificity, lift, support_count in candidates:
        score = (confidence, specificity, lift, support_count)
        if skill not in best or score > best[skill]:
            best[skill] = score
    rule_skills = [skill for skill, _ in sorted(best.items(), key=lambda item: item[1], reverse=True)]
    result = rule_skills[:k]
    source = ["Luật Apriori"] * len(result)
    seen = known | set(result)
    for skill in model["popularity"].get(role_exp, []):
        if len(result) >= k: break
        if skill not in seen:
            result.append(skill)
            source.append("Phổ biến theo role + exp")
            seen.add(skill)
    return pd.DataFrame({"Hạng": range(1, len(result)+1), "Technical skill": result, "Nguồn gợi ý": source})


## 3. Chọn thông tin đầu vào

In [3]:
role_exp_pairs = [value.split(" | ", 1) for value in model["role_exp_values"] if " | " in value]
experience_by_role = {}
for role, exp in role_exp_pairs:
    experience_by_role.setdefault(role, []).append(exp)
role_picker = widgets.Dropdown(options=sorted(experience_by_role), description="Role:")
exp_picker = widgets.Dropdown(options=sorted(experience_by_role[role_picker.value]), description="Exp:")
skill_picker = widgets.SelectMultiple(options=model["skills"], description="Đã biết:", rows=12)
k_picker = widgets.IntSlider(value=10, min=3, max=20, description="Số gợi ý:")
button = widgets.Button(description="Gợi ý skill", button_style="primary")
output = widgets.Output()
def update_experience(change):
    exp_picker.options = sorted(experience_by_role[change["new"]])
    exp_picker.value = exp_picker.options[0]
role_picker.observe(update_experience, names="value")
def show_recommendations(_):
    with output:
        output.clear_output()
        display(recommend(role_picker.value, exp_picker.value, skill_picker.value, k_picker.value))
button.on_click(show_recommendations)
display(widgets.VBox([role_picker, exp_picker, skill_picker, k_picker, button, output]))


### Lưu ý khi demo

Model được fit từ dữ liệu VietJobs_IT đã tiền xử lý. Kết quả nên được trình bày là gợi ý skill dựa trên đồng xuất hiện trong tin tuyển dụng, không phải lộ trình học được xác nhận theo thời gian. Nhóm role–exp ít hoặc không có luật có thể nhận nhiều gợi ý từ fallback phổ biến.